# Patient embeddings and supervised symptom training

This notebook is designed to run top to bottom in Google Colab. Select **Runtime → Change runtime type → GPU** before starting.

**Cells marked EDIT ME require configuration.** Embedding generation and supervised training are separate sections: after embeddings exist in Google Drive, rerun only the loading and training sections for new experiments. The notebook never trains ModernBERT itself.

## 1. Check the Colab GPU

Embedding generation later requires CUDA. The supervised model automatically uses CUDA when available and otherwise falls back to CPU.

In [ ]:
import platform
import torch

CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "CPU"
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {CUDA_AVAILABLE}")
print(f"Detected device: {DEVICE_NAME}")
if not CUDA_AVAILABLE:
    print("WARNING: Enable a GPU runtime before running the embedding section.")

## 2. Clone or update the project repository — EDIT ME

Replace `YOUR_USERNAME/YOUR_REPOSITORY` with the GitHub repository containing `prepare_patient_embeddings.py` and `requirements.txt`. For a private repository, configure Colab GitHub credentials before running this cell.

In [ ]:
from pathlib import Path
import subprocess

# EDIT ME: replace this placeholder with the real repository URL.
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPOSITORY.git"
PROJECT_DIR = Path("/content/Text2DAG")

if "YOUR_USERNAME/YOUR_REPOSITORY" in REPO_URL:
    raise ValueError("Edit REPO_URL before running this cell.")

if (PROJECT_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True
    )
elif PROJECT_DIR.exists():
    raise RuntimeError(f"{PROJECT_DIR} exists but is not a Git repository.")
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

print(f"Project ready at {PROJECT_DIR}")

## 3. Install dependencies

This installs the repository requirements plus scikit-learn for splitting and evaluation.

In [ ]:
import sys

requirements_path = PROJECT_DIR / "requirements.txt"
if not requirements_path.is_file():
    raise FileNotFoundError(f"Missing requirements file: {requirements_path}")
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "-r", str(requirements_path), "scikit-learn>=1.4,<2"
    ],
    check=True,
)
print("Dependencies installed.")

## 4. Mount Google Drive

Approve the Google Drive authorization prompt. All generated artifacts are stored in Drive so they survive Colab runtime resets.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 5. Configure Google Drive paths — EDIT ME

Edit `DRIVE_ROOT`, `MAPPING_PATH`, and `SOURCE_PATH` to match your Drive layout. The embedding, checkpoint, and evaluation directories are configurable independently. Set `FORCE_RECOMPUTE=True` only when existing embeddings should be replaced.

In [ ]:
# EDIT ME: configure these Google Drive paths.
DRIVE_ROOT = Path("/content/drive/MyDrive/Text2DAG")
MAPPING_PATH = DRIVE_ROOT / "inputs/subsentence_mapping.csv"
SOURCE_PATH = DRIVE_ROOT / "inputs/SynSUM.csv"
EMBEDDING_OUTPUT_DIR = DRIVE_ROOT / "generated_embeddings"
MODEL_CHECKPOINT_DIR = DRIVE_ROOT / "model_checkpoints"
EVALUATION_RESULTS_DIR = DRIVE_ROOT / "evaluation_results"

FORCE_RECOMPUTE = False
EMBEDDING_BATCH_SIZE = 32
EMBEDDING_MODEL_NAME = "nomic-ai/modernbert-embed-base"
RANDOM_SEED = 42

for directory in [EMBEDDING_OUTPUT_DIR, MODEL_CHECKPOINT_DIR, EVALUATION_RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
for input_path in [MAPPING_PATH, SOURCE_PATH]:
    if not input_path.is_file():
        raise FileNotFoundError(
            f"Input not found: {input_path}. Edit the path configuration cell."
        )

EMBEDDING_NPZ_PATH = EMBEDDING_OUTPUT_DIR / "patient_embeddings_and_labels.npz"
print(f"Mapping: {MAPPING_PATH}")
print(f"SynSUM: {SOURCE_PATH}")
print(f"Embeddings: {EMBEDDING_OUTPUT_DIR}")
print(f"Checkpoints: {MODEL_CHECKPOINT_DIR}")
print(f"Evaluation results: {EVALUATION_RESULTS_DIR}")

# Part A — ModernBERT embedding generation

This section embeds each valid subsentence, directly mean-pools by patient `data_row_idx`, and saves the patient arrays and traceability files to Google Drive. It is skipped when the NPZ already exists unless `FORCE_RECOMPUTE` is true.

In [ ]:
if EMBEDDING_NPZ_PATH.exists() and not FORCE_RECOMPUTE:
    print(f"Skipping embedding generation; output exists: {EMBEDDING_NPZ_PATH}")
else:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is required for this notebook's embedding step. "
            "Select Runtime > Change runtime type > GPU and rerun."
        )
    preprocessing_script = PROJECT_DIR / "prepare_patient_embeddings.py"
    if not preprocessing_script.is_file():
        raise FileNotFoundError(f"Missing preprocessing script: {preprocessing_script}")

    command = [
        sys.executable,
        str(preprocessing_script),
        "--mapping-path", str(MAPPING_PATH),
        "--source-path", str(SOURCE_PATH),
        "--output-dir", str(EMBEDDING_OUTPUT_DIR),
        "--model-name", EMBEDDING_MODEL_NAME,
        "--batch-size", str(EMBEDDING_BATCH_SIZE),
        "--device", "cuda",
    ]
    print("Running:", " ".join(command))
    subprocess.run(command, check=True, cwd=PROJECT_DIR)

if not EMBEDDING_NPZ_PATH.is_file():
    raise FileNotFoundError(f"Expected embedding output was not created: {EMBEDDING_NPZ_PATH}")
print(f"Embedding arrays ready: {EMBEDDING_NPZ_PATH}")

## Load and validate the saved arrays

Start from this cell when rerunning only supervised training in a later Colab session (after mounting Drive and configuring the paths).

In [ ]:
import numpy as np

with np.load(EMBEDDING_NPZ_PATH, allow_pickle=False) as saved:
    required_keys = {"patient_ids", "X", "y", "label_names"}
    missing_keys = required_keys - set(saved.files)
    if missing_keys:
        raise ValueError(f"Embedding NPZ is missing keys: {sorted(missing_keys)}")
    patient_ids = saved["patient_ids"].astype(np.int64, copy=False)
    X = saved["X"].astype(np.float32, copy=False)
    y = saved["y"].astype(np.int64, copy=False)
    label_names = saved["label_names"].astype(str)

EXPECTED_LABEL_NAMES = np.array(["dysp", "cough", "pain", "fever", "nasal"])
if X.ndim != 2 or X.shape[1] != 768:
    raise ValueError(f"Expected X shape (n_patients, 768), found {X.shape}.")
if y.ndim != 2 or y.shape[1] != 5:
    raise ValueError(f"Expected y shape (n_patients, 5), found {y.shape}.")
if X.shape[0] != y.shape[0] or X.shape[0] != len(patient_ids):
    raise ValueError(
        f"Row mismatch: X={X.shape}, y={y.shape}, patient_ids={patient_ids.shape}."
    )
if not np.array_equal(label_names, EXPECTED_LABEL_NAMES):
    raise ValueError(f"Unexpected label order: {label_names.tolist()}")
if len(np.unique(patient_ids)) != len(patient_ids):
    raise ValueError("Duplicate patient IDs found in saved arrays.")
if not np.isfinite(X).all():
    raise ValueError("X contains NaN or infinite values.")
if not np.isfinite(y).all():
    raise ValueError("y contains NaN or infinite values.")
for index in [0, 1, 2, 4]:
    if not set(np.unique(y[:, index])).issubset({0, 1}):
        raise ValueError(f"Binary label {label_names[index]} contains invalid values.")
if not set(np.unique(y[:, 3])).issubset({0, 1, 2}):
    raise ValueError("Fever contains values outside {0, 1, 2}.")

print(f"patient_ids: {patient_ids.shape}, {patient_ids.dtype}")
print(f"X: {X.shape}, {X.dtype}")
print(f"y: {y.shape}, {y.dtype}")
print(f"label_names: {label_names.tolist()}")

# Part B — Supervised multi-task symptom training

This section uses only the saved patient arrays; it does not recompute ModernBERT embeddings. The requested labels are not all binary: `fever` is `none=0`, `low=1`, `high=2`. Therefore, the model uses four binary heads (dysp, cough, pain, nasal) and one three-class fever head. Binary cross-entropy is used for the four binary tasks and cross-entropy for fever.

## Training configuration — optionally edit

Change hyperparameters here for a new experiment. Keep a new `RUN_NAME` if you want to preserve artifacts from earlier runs.

In [ ]:
import json
import os
import random
from datetime import datetime, timezone

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

RUN_NAME = "symptom_mlp_seed42"
TRAINING_CONFIG = {
    "random_seed": RANDOM_SEED,
    "hidden_dim": 256,
    "dropout": 0.25,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "batch_size": 32,
    "max_epochs": 100,
    "early_stopping_patience": 12,
    "validation_fraction": 0.15,
    "test_fraction": 0.15,
    "binary_threshold": 0.5,
}

def seed_everything(seed: int) -> None:
    """Seed Python, NumPy, and PyTorch for reproducible experiments."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(RANDOM_SEED)
TRAIN_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_CHECKPOINT_DIR = MODEL_CHECKPOINT_DIR / RUN_NAME
RUN_RESULTS_DIR = EVALUATION_RESULTS_DIR / RUN_NAME
RUN_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RUN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BEST_CHECKPOINT_PATH = RUN_CHECKPOINT_DIR / "best_model.pt"

run_metadata = {
    **TRAINING_CONFIG,
    "run_name": RUN_NAME,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "embedding_model": EMBEDDING_MODEL_NAME,
    "embedding_npz": str(EMBEDDING_NPZ_PATH),
    "label_names": label_names.tolist(),
    "training_device": str(TRAIN_DEVICE),
}
with open(RUN_CHECKPOINT_DIR / "training_config.json", "w", encoding="utf-8") as handle:
    json.dump(run_metadata, handle, indent=2)
with open(RUN_CHECKPOINT_DIR / "random_seed.txt", "w", encoding="utf-8") as handle:
    handle.write(f"{RANDOM_SEED}\n")

print(f"Training device: {TRAIN_DEVICE}")
print(f"Run checkpoint directory: {RUN_CHECKPOINT_DIR}")
print(f"Run results directory: {RUN_RESULTS_DIR}")

## Create reproducible train, validation, and test splits

Normalization statistics are learned from the training split only and are stored in the best checkpoint.

In [ ]:
all_indices = np.arange(len(patient_ids))
train_val_indices, test_indices = train_test_split(
    all_indices,
    test_size=TRAINING_CONFIG["test_fraction"],
    random_state=RANDOM_SEED,
    shuffle=True,
)
validation_share_of_train_val = (
    TRAINING_CONFIG["validation_fraction"]
    / (1.0 - TRAINING_CONFIG["test_fraction"])
)
train_indices, validation_indices = train_test_split(
    train_val_indices,
    test_size=validation_share_of_train_val,
    random_state=RANDOM_SEED,
    shuffle=True,
)

feature_mean = X[train_indices].mean(axis=0, dtype=np.float64).astype(np.float32)
feature_std = X[train_indices].std(axis=0, dtype=np.float64).astype(np.float32)
feature_std[feature_std < 1e-8] = 1.0
X_scaled = ((X - feature_mean) / feature_std).astype(np.float32)

def make_loader(indices: np.ndarray, shuffle: bool) -> DataLoader:
    dataset = TensorDataset(
        torch.from_numpy(X_scaled[indices]),
        torch.from_numpy(y[indices]),
    )
    generator = torch.Generator().manual_seed(RANDOM_SEED)
    return DataLoader(
        dataset,
        batch_size=TRAINING_CONFIG["batch_size"],
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

train_loader = make_loader(train_indices, shuffle=True)
validation_loader = make_loader(validation_indices, shuffle=False)
test_loader = make_loader(test_indices, shuffle=False)

np.savez_compressed(
    RUN_RESULTS_DIR / "data_split_patient_ids.npz",
    train_patient_ids=patient_ids[train_indices],
    validation_patient_ids=patient_ids[validation_indices],
    test_patient_ids=patient_ids[test_indices],
)
print(
    f"Split sizes — train: {len(train_indices)}, "
    f"validation: {len(validation_indices)}, test: {len(test_indices)}"
)

## Define the multi-task model, losses, and metrics

In [ ]:
BINARY_LABEL_INDICES = [0, 1, 2, 4]
BINARY_LABEL_NAMES = [label_names[index] for index in BINARY_LABEL_INDICES]
FEVER_INDEX = 3

class SymptomMultiTaskMLP(nn.Module):
    """Shared MLP with four binary outputs and one 3-class fever output."""

    def __init__(self, input_dim: int, hidden_dim: int, dropout: float) -> None:
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.binary_head = nn.Linear(hidden_dim // 2, len(BINARY_LABEL_INDICES))
        self.fever_head = nn.Linear(hidden_dim // 2, 3)

    def forward(self, features: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        shared = self.shared(features)
        return self.binary_head(shared), self.fever_head(shared)

binary_train_targets = y[train_indices][:, BINARY_LABEL_INDICES]
positive_counts = binary_train_targets.sum(axis=0).astype(np.float32)
negative_counts = len(binary_train_targets) - positive_counts
binary_pos_weight = torch.from_numpy(
    negative_counts / np.maximum(positive_counts, 1.0)
).to(TRAIN_DEVICE)
fever_counts = np.bincount(y[train_indices, FEVER_INDEX], minlength=3).astype(np.float32)
fever_class_weight = torch.from_numpy(
    len(train_indices) / (3.0 * np.maximum(fever_counts, 1.0))
).to(TRAIN_DEVICE)

binary_loss_function = nn.BCEWithLogitsLoss(pos_weight=binary_pos_weight)
fever_loss_function = nn.CrossEntropyLoss(weight=fever_class_weight)

def calculate_metrics(
    targets: np.ndarray,
    binary_probabilities: np.ndarray,
    fever_probabilities: np.ndarray,
    loss: float,
) -> tuple[dict, np.ndarray]:
    threshold = TRAINING_CONFIG["binary_threshold"]
    binary_predictions = (binary_probabilities >= threshold).astype(np.int64)
    fever_predictions = fever_probabilities.argmax(axis=1).astype(np.int64)
    predictions = np.zeros_like(targets, dtype=np.int64)
    predictions[:, BINARY_LABEL_INDICES] = binary_predictions
    predictions[:, FEVER_INDEX] = fever_predictions

    per_label = {}
    task_f1_scores = []
    for output_index, label_index in enumerate(BINARY_LABEL_INDICES):
        truth = targets[:, label_index]
        prediction = binary_predictions[:, output_index]
        precision, recall, f1, _ = precision_recall_fscore_support(
            truth, prediction, average="binary", zero_division=0
        )
        try:
            auroc = float(roc_auc_score(truth, binary_probabilities[:, output_index]))
        except ValueError:
            auroc = None
        per_label[str(label_names[label_index])] = {
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "auroc": auroc,
            "accuracy": float(accuracy_score(truth, prediction)),
        }
        task_f1_scores.append(float(f1))

    fever_truth = targets[:, FEVER_INDEX]
    precision, recall, fever_f1, _ = precision_recall_fscore_support(
        fever_truth, fever_predictions, average="macro", zero_division=0
    )
    try:
        fever_auroc = float(
            roc_auc_score(
                fever_truth, fever_probabilities, multi_class="ovr", average="macro"
            )
        )
    except ValueError:
        fever_auroc = None
    per_label["fever"] = {
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "f1_macro": float(fever_f1),
        "auroc_ovr_macro": fever_auroc,
        "accuracy": float(accuracy_score(fever_truth, fever_predictions)),
    }
    task_f1_scores.append(float(fever_f1))
    metrics = {
        "loss": float(loss),
        "overall_macro_f1": float(np.mean(task_f1_scores)),
        "per_label": per_label,
    }
    return metrics, predictions

def run_epoch(
    model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer | None = None
) -> tuple[dict, dict]:
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    all_targets, all_binary_probabilities, all_fever_probabilities = [], [], []

    for features, targets in loader:
        features = features.to(TRAIN_DEVICE, non_blocking=True)
        targets = targets.to(TRAIN_DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            binary_logits, fever_logits = model(features)
            binary_targets = targets[:, BINARY_LABEL_INDICES].float()
            fever_targets = targets[:, FEVER_INDEX].long()
            loss = (
                binary_loss_function(binary_logits, binary_targets)
                + fever_loss_function(fever_logits, fever_targets)
            )
            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * len(features)
        all_targets.append(targets.detach().cpu().numpy())
        all_binary_probabilities.append(torch.sigmoid(binary_logits).detach().cpu().numpy())
        all_fever_probabilities.append(torch.softmax(fever_logits, dim=1).detach().cpu().numpy())

    targets_array = np.concatenate(all_targets)
    binary_probabilities = np.concatenate(all_binary_probabilities)
    fever_probabilities = np.concatenate(all_fever_probabilities)
    mean_loss = total_loss / len(loader.dataset)
    metrics, predictions = calculate_metrics(
        targets_array, binary_probabilities, fever_probabilities, mean_loss
    )
    arrays = {
        "targets": targets_array,
        "predictions": predictions,
        "binary_probabilities": binary_probabilities,
        "fever_probabilities": fever_probabilities,
    }
    return metrics, arrays

## Train with early stopping and save the best checkpoint

The checkpoint is selected by validation macro-F1 averaged across the four binary tasks and multiclass fever.

In [ ]:
model = SymptomMultiTaskMLP(
    input_dim=X.shape[1],
    hidden_dim=TRAINING_CONFIG["hidden_dim"],
    dropout=TRAINING_CONFIG["dropout"],
).to(TRAIN_DEVICE)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=TRAINING_CONFIG["learning_rate"],
    weight_decay=TRAINING_CONFIG["weight_decay"],
)

best_validation_score = -np.inf
epochs_without_improvement = 0
history = []

for epoch in range(1, TRAINING_CONFIG["max_epochs"] + 1):
    train_metrics, _ = run_epoch(model, train_loader, optimizer)
    validation_metrics, _ = run_epoch(model, validation_loader)
    validation_score = validation_metrics["overall_macro_f1"]
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_macro_f1": train_metrics["overall_macro_f1"],
            "validation_loss": validation_metrics["loss"],
            "validation_macro_f1": validation_score,
        }
    )
    print(
        f"Epoch {epoch:03d} | train loss {train_metrics['loss']:.4f} | "
        f"val loss {validation_metrics['loss']:.4f} | val macro-F1 {validation_score:.4f}"
    )

    if validation_score > best_validation_score:
        best_validation_score = validation_score
        epochs_without_improvement = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": epoch,
                "best_validation_macro_f1": best_validation_score,
                "training_config": TRAINING_CONFIG,
                "random_seed": RANDOM_SEED,
                "label_names": label_names.tolist(),
                "feature_mean": feature_mean,
                "feature_std": feature_std,
            },
            BEST_CHECKPOINT_PATH,
        )
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= TRAINING_CONFIG["early_stopping_patience"]:
            print(f"Early stopping at epoch {epoch}.")
            break

history_frame = pd.DataFrame(history)
history_frame.to_csv(RUN_RESULTS_DIR / "training_history.csv", index=False)
if not BEST_CHECKPOINT_PATH.is_file():
    raise RuntimeError("Training finished without creating a best checkpoint.")
print(f"Best validation macro-F1: {best_validation_score:.4f}")
print(f"Best checkpoint: {BEST_CHECKPOINT_PATH}")

## Evaluate the best checkpoint and save predictions and metrics

This cell reloads the best validation checkpoint, evaluates the held-out test set, and writes JSON metrics plus compressed NumPy predictions to Google Drive.

In [ ]:
checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=TRAIN_DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
test_metrics, test_arrays = run_epoch(model, test_loader)

metrics_payload = {
    "run_name": RUN_NAME,
    "random_seed": RANDOM_SEED,
    "best_epoch": int(checkpoint["epoch"]),
    "best_validation_macro_f1": float(checkpoint["best_validation_macro_f1"]),
    "test": test_metrics,
}
with open(RUN_RESULTS_DIR / "evaluation_metrics.json", "w", encoding="utf-8") as handle:
    json.dump(metrics_payload, handle, indent=2)

np.savez_compressed(
    RUN_RESULTS_DIR / "test_predictions.npz",
    patient_ids=patient_ids[test_indices],
    y_true=test_arrays["targets"].astype(np.int64),
    y_pred=test_arrays["predictions"].astype(np.int64),
    binary_probabilities=test_arrays["binary_probabilities"].astype(np.float32),
    binary_label_names=np.asarray(BINARY_LABEL_NAMES),
    fever_probabilities=test_arrays["fever_probabilities"].astype(np.float32),
    fever_class_names=np.asarray(["none", "low", "high"]),
    label_names=label_names,
)

print(json.dumps(metrics_payload, indent=2))
print(f"Metrics saved to: {RUN_RESULTS_DIR / 'evaluation_metrics.json'}")
print(f"Predictions saved to: {RUN_RESULTS_DIR / 'test_predictions.npz'}")

## Saved artifacts

The embedding directory contains the patient CSV, Parquet, NPZ, and subsentence traceability Parquet. The run checkpoint directory contains the best model, training configuration, and random seed. The run evaluation directory contains split patient IDs, training history, test predictions and probabilities, and evaluation metrics.